# 01 — Download audio from Xeno-canto

This notebook demonstrates how to use `src/download.py` to download `.mp3` audio recordings from the [Xeno-canto](https://xeno-canto.org/) API and organise them under `data/raw/<species>/`.

**Pipeline:**
1. Environment setup (Colab / local)
2. API key and species list configuration
3. Search recordings
4. Download audio
5. Verify downloaded files

## 1. Environment setup

The cell below automatically detects Google Colab and, if running there, clones the repository and installs all dependencies.

In [ ]:
import sys
import os

# Detect Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Google Drive (optional — for persistent storage)
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone the repository if not already present
    repo_dir = '/content/bird-acoustics-classifier'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/danort92/bird-acoustics-classifier.git {repo_dir}

    os.chdir(repo_dir)
    !pip install -q -r requirements.txt
    print('Colab setup complete.')
else:
    # Local: ensure we are at the project root
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(project_root)
    print(f'Working directory: {os.getcwd()}')

## 2. Configuration

Set the **Xeno-canto API key** and the **species list** to download.

> **API key**: optional for the public v2 endpoint. If you have a registered key,
> set it via the `XENO_CANTO_API_KEY` environment variable or enter it when prompted
> by the module.

In [ ]:
# Option A — environment variable (recommended)
# os.environ['XENO_CANTO_API_KEY'] = 'your_key_here'

# Species list (scientific names)
SPECIES = [
    "Turdus merula",       # Common blackbird
    "Erithacus rubecula",  # European robin
    "Parus major",         # Great tit
]

MAX_PER_SPECIES = 30   # Maximum number of files per species
QUALITY = "A"          # Minimum quality rating (A = best, E = worst)
OUTPUT_DIR = "data/raw"

print(f"Selected species: {SPECIES}")
print(f"Max recordings per species: {MAX_PER_SPECIES}")
print(f"Minimum quality: {QUALITY}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Initialise the downloader

In [ ]:
from src.download import XenoCantoDownloader

# The constructor automatically reads XENO_CANTO_API_KEY
# or prompts interactively if not set.
downloader = XenoCantoDownloader(
    output_dir=OUTPUT_DIR,
    request_delay=0.5,  # pause between requests (seconds)
)
print("Downloader initialised.")

## 4. Search recordings (preview — no download)

Before downloading, check how many recordings are available for each species.

In [ ]:
import pandas as pd

preview_rows = []

for species in SPECIES:
    recordings = downloader.search_species(species, quality=QUALITY, max_results=MAX_PER_SPECIES)
    for r in recordings[:3]:  # show only the first 3 for brevity
        preview_rows.append({
            "species": species,
            "id": r.get("id"),
            "country": r.get("cnt"),
            "quality": r.get("q"),
            "length": r.get("length"),
            "file_url": r.get("file"),
        })
    print(f"{species}: {len(recordings)} recordings found")

df_preview = pd.DataFrame(preview_rows)
df_preview

## 5. Download audio

Download all recordings for the configured species and save them to `data/raw/<species>/`.

In [ ]:
results = downloader.download_species(
    species_list=SPECIES,
    max_per_species=MAX_PER_SPECIES,
    quality=QUALITY,
)

print("\n=== Download summary ===")
for species, paths in results.items():
    print(f"  {species}: {len(paths)} files downloaded")

## 6. Verify downloaded files

In [ ]:
from pathlib import Path

downloaded = downloader.list_downloaded()

summary_rows = []
for species_dir, files in downloaded.items():
    total_mb = sum(f.stat().st_size for f in files) / 1_048_576
    summary_rows.append({
        "species_dir": species_dir,
        "n_files": len(files),
        "total_MB": round(total_mb, 2),
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
print(f"\nTotal files: {df_summary['n_files'].sum()}")
print(f"Total size: {df_summary['total_MB'].sum():.2f} MB")

## 7. Resulting directory structure

Print the directory tree after the download.

In [ ]:
def print_tree(root: Path, max_files: int = 5) -> None:
    """Print a compact directory tree."""
    print(f"{root}/")
    for species_dir in sorted(root.iterdir()):
        if not species_dir.is_dir():
            continue
        files = sorted(species_dir.glob('*.mp3'))
        print(f"  {species_dir.name}/  ({len(files)} files)")
        for f in files[:max_files]:
            print(f"    {f.name}")
        if len(files) > max_files:
            print(f"    ... and {len(files) - max_files} more files")

print_tree(Path(OUTPUT_DIR))

---
**Next step:** `02_preprocessing.ipynb` — convert audio to mel spectrograms